# DeepSpot-M on Google Colab

Part of [tnbc-research](https://github.com/olimpijada72/tnbc-research). Predicts
spatial gene expression from H&E whole-slide images with
[DeepSpot-M](https://huggingface.co/ratschlab/DeepSpotM) on a Colab GPU.

The code comes from the copy of DeepSpot-M pinned in `external/DeepSpotM/` of that
repo, and the weights from a fixed Hugging Face revision, so upstream changes
do not affect this notebook.

**Before you start (once):**

1. **GPU:** *Runtime → Change runtime type → T4 GPU* (or L4 / A100 on Colab Pro).
2. **Model access:** open [ratschlab/DeepSpotM](https://huggingface.co/ratschlab/DeepSpotM)
   while logged in to Hugging Face and accept the terms (the model is gated).
3. **Token:** create a *read* token at https://huggingface.co/settings/tokens. In Colab,
   click the key icon in the left sidebar, add a secret named `HF_TOKEN` with the token
   as its value, and turn on *Notebook access*.
4. **Slides:** put your `.svs` / `.tif` / `.ndpi` slides in a folder on Google Drive
   (default: `MyDrive/deepspot/slides`). They should be ~20x (~0.5 µm/px).

**Then:** set the options in the next cell and choose *Runtime → Run all*.
Results are written to Google Drive as one `.h5ad` per slide, so they survive a
disconnect. If the session drops, run all again: finished slides are skipped.

In [ ]:
# @title Settings
# Genes to predict, separated by spaces. Leave empty for the full ~19k-gene panel
# (much larger output files and needs more RAM).
GENES = "ESR1 PGR ERBB2 MKI67 EPCAM PTPRC COL1A1"  # @param {type:"string"}
# Folder on Google Drive with your slides, and where to write the results.
SLIDES_DIR = "/content/drive/MyDrive/deepspot/slides"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/deepspot/output"  # @param {type:"string"}
# Tiles per forward pass. Lower it if you get a CUDA out-of-memory error.
BATCH_SIZE = 64  # @param {type:"integer"}
# Run the quick test on a small demo slide before your own slides.
RUN_DEMO = True  # @param {type:"boolean"}
# Keep the 5.4 GB model on Drive so later sessions skip the download
# (uses Drive space).
CACHE_MODEL_ON_DRIVE = False  # @param {type:"boolean"}
# Branch or tag of tnbc-research to run. Use a tag (e.g. v0.1) for results you
# want to reproduce later.
REPO_REF = "main"  # @param {type:"string"}
# Hugging Face commit of ratschlab/DeepSpotM to load. Only change it on purpose.
HF_REVISION = "48be27af436a50e5c74175680ac2b7b2596a506b"  # @param {type:"string"}

## 1. Install

In [ ]:
# @title Check the GPU and install DeepSpot-M (about 2-3 minutes)
import os
import torch

assert torch.cuda.is_available(), (
    "No GPU found. Choose Runtime > Change runtime type > T4 GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

# libvips reads the slides. The Ubuntu package includes OpenSlide (.svs, .ndpi, ...).
!apt-get -qq update > /dev/null && apt-get -qq install -y libvips-dev > /dev/null

REPO = "/content/tnbc-research"
if not os.path.exists(REPO):
    !git clone -q --depth 1 --branch {REPO_REF} https://github.com/olimpijada72/tnbc-research {REPO}
DEEPSPOT = f"{REPO}/external/DeepSpotM"

# Keep Colab's own torch. transformers must stay below 5: with 5.x, loading the
# model fails with "Target modules {'query', 'key', 'value'} not found".
!cd {DEEPSPOT} && pip install -q ".[wsi]" "transformers>=4.43,<5" matplotlib
# Colab ships torchao 0.10, and recent peft refuses to build LoRA layers when an
# older torchao (<0.16) is installed. DeepSpot-M does not use torchao.
!pip uninstall -q -y torchao
# The apt libvips pulls in the system HDF5 1.10, while h5py (used to write .h5ad)
# bundles HDF5 1.14. Whichever loads first wins, and h5py crashes with
# "Not a datatype" if pyvips loads first. So import h5py before pyvips, both
# here and in the prediction script (predict_wsi.py imports pyvips first).
PREDICT = ("python -c 'import h5py, runpy, sys; "
           'sys.argv[0] = "predict_wsi.py"; '
           'runpy.run_path("examples/predict_wsi.py", run_name="__main__")' + "'")
print("installed")

## 2. Hugging Face token and Google Drive

In [ ]:
# @title Log in and mount Drive
from google.colab import drive, userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as err:
    raise RuntimeError(
        "No Hugging Face token. Add a Colab secret named HF_TOKEN (key icon in the "
        "left sidebar) and turn on notebook access.") from err

from huggingface_hub import auth_check
auth_check("ratschlab/DeepSpotM")  # fails here if you have not accepted the model terms
print("Hugging Face access OK")

drive.mount("/content/drive")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download the pinned model revision; predict_wsi.py then loads it from this folder.
# Passed as cache_dir because huggingface_hub reads HF_HOME only at import.
CACHE_DIR = "/content/drive/MyDrive/deepspot/hf_cache" if CACHE_MODEL_ON_DRIVE else None
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download(
    "ratschlab/DeepSpotM", revision=HF_REVISION, cache_dir=CACHE_DIR,
    allow_patterns=["config.json", "model.safetensors", "tokens.csv"])
print("model:", MODEL_DIR)
GENE_ARGS = f"--genes {GENES}" if GENES.strip() else ""
MODEL_ARGS = f"--repo {MODEL_DIR}"

In [ ]:
# @title Plotting helper
import h5py  # noqa: F401  must load before pyvips, see the install cell
import numpy as np
import matplotlib.pyplot as plt


def show(h5ad_path, slide_path, genes=None, max_genes=7):
    """Plot the H&E next to one predicted-expression map per gene."""
    import anndata as ad
    import pyvips

    a = ad.read_h5ad(h5ad_path)
    genes = list(genes or a.var_names[:max_genes])
    tile = a.uns["deepspotm"]["tile_px"]

    slide = pyvips.Image.new_from_file(str(slide_path))
    ov = pyvips.Image.thumbnail(str(slide_path), 1500)
    he = np.ndarray(buffer=ov.write_to_memory(), dtype=np.uint8,
                    shape=[ov.height, ov.width, ov.bands])[..., :3]
    nx, ny = slide.width // tile, slide.height // tile
    extent = (0, nx * tile * ov.width / slide.width,
              ny * tile * ov.height / slide.height, 0)
    cols, rows = a.obs["x"].to_numpy() // tile, a.obs["y"].to_numpy() // tile

    n = len(genes) + 1
    ncol = min(n, 4)
    fig, axes = plt.subplots(-(-n // ncol), ncol, figsize=(5 * ncol, 5 * -(-n // ncol)),
                             squeeze=False)
    axes = axes.ravel()
    axes[0].imshow(he)
    axes[0].set_title("H&E")
    for ax, g in zip(axes[1:], genes):
        grid = np.full((ny, nx), np.nan)
        grid[rows, cols] = np.asarray(a[:, g].X).ravel()
        ax.imshow(he, alpha=0.3)
        im = ax.imshow(grid, cmap="viridis", extent=extent, interpolation="nearest")
        ax.set_title(g)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{a.uns['deepspotm']['slide']}: {a.n_obs} tiles, "
                 f"loader {slide.get('vips-loader')}")
    plt.tight_layout()
    plt.show()

## 3. Quick test on a demo slide

Downloads a small (~26 MB) open-access TCGA breast cancer slide and predicts 7 marker
genes for every tissue tile. This checks the whole setup in a few minutes and
tells you the speed, so you can estimate how long your slides will take.

In the maps, PTPRC (immune cells) should be high over lymphocyte clusters, COL1A1
(collagen) over stroma, and EPCAM / ESR1 over the tumour epithelium. If every map
looks like flat noise, something is wrong.

In [ ]:
# @title Run the demo
import hashlib
import time
import urllib.request

if RUN_DEMO:
    DEMO = "/content/demo/TCGA-OL-A5RW-01Z-00-DX1.svs"
    if not os.path.exists(DEMO):
        os.makedirs("/content/demo", exist_ok=True)
        urllib.request.urlretrieve(
            "https://api.gdc.cancer.gov/data/f2d5aa37-d9ce-4264-a447-fc69dd0d7d85", DEMO)
    with open(DEMO, "rb") as fh:
        assert hashlib.md5(fh.read()).hexdigest() == "1a169f76fc7d21e767a93225968b9531", \
            "demo slide checksum mismatch; delete it and run again"

    t0 = time.time()
    !cd {DEEPSPOT} && {PREDICT} {DEMO} -o /content/demo/demo.h5ad {MODEL_ARGS} \
        --genes ESR1 PGR ERBB2 MKI67 EPCAM PTPRC COL1A1 --batch-size {BATCH_SIZE}
    if _exit_code:
        raise RuntimeError("prediction failed, see the error above")
    print(f"took {time.time() - t0:.0f}s, including the model load")
    show("/content/demo/demo.h5ad", DEMO)

## 4. Your slides

Copies the slides from Drive to the Colab disk (reading from Drive directly is slow),
skips slides that already have a result in `OUTPUT_DIR`, and writes one
`<slide name>.h5ad` per slide to `OUTPUT_DIR`.

A whole slide can have tens of thousands of tissue tiles, so this can take a while.
Keep the browser tab open. On the free tier, sessions end after a few hours of
running or when idle.

In [ ]:
# @title Run all slides in SLIDES_DIR
import shlex
import shutil
from pathlib import Path

import pyvips

SUFFIXES = {".svs", ".tif", ".tiff", ".ndpi", ".scn", ".svslide", ".bif"}
slides = sorted(p for p in Path(SLIDES_DIR).rglob("*") if p.suffix.lower() in SUFFIXES)
todo = [p for p in slides if not (Path(OUTPUT_DIR) / f"{p.stem}.h5ad").exists()]
print(f"{len(slides)} slides in {SLIDES_DIR}, {len(slides) - len(todo)} already done, "
      f"{len(todo)} to run")

local_dir = Path("/content/slides")
local_dir.mkdir(exist_ok=True)
local = []
for p in todo:
    dest = local_dir / p.name
    if not dest.exists() or dest.stat().st_size != p.stat().st_size:
        print("copying", p.name)
        shutil.copy(p, dest)
    # DeepSpot-M expects ~20x (~0.5 um/px); tiles are cut at native resolution.
    img = pyvips.Image.new_from_file(str(dest))
    mpp = next((img.get(f) for f in ("openslide.mpp-x", "aperio.MPP")
                if f in img.get_fields()), None)
    if mpp is None and img.get("xres") > 1:  # plain TIFF: xres is pixels per mm
        mpp = f"{1000 / img.get('xres'):.3f}"
    note = "" if mpp is None or float(mpp) > 0.35 else "  <- looks like 40x, results may be off"
    print(f"  {p.name}: {mpp or 'unknown'} um/px{note}")
    local.append(dest)

if local:
    paths = " ".join(shlex.quote(str(p)) for p in local)
    !cd {DEEPSPOT} && {PREDICT} {paths} -o {shlex.quote(OUTPUT_DIR)} {MODEL_ARGS} \
        {GENE_ARGS} --batch-size {BATCH_SIZE}
    if _exit_code:
        raise RuntimeError("prediction failed, see the error above")

## 5. Look at a result

In [ ]:
# @title Plot one slide
SLIDE_NAME = ""  # @param {type:"string"}
# Leave SLIDE_NAME empty to plot the first slide. For a full-panel run, pass
# genes=[...] to show() to choose which genes to plot.
done = [p for p in slides if (Path(OUTPUT_DIR) / f"{p.stem}.h5ad").exists()]
pick = next((p for p in done if p.name == SLIDE_NAME or p.stem == SLIDE_NAME),
            done[0] if done else None)
if pick is None:
    print("no results yet")
else:
    show(Path(OUTPUT_DIR) / f"{pick.stem}.h5ad", pick)